# moomoo User ID Batch Extractor

这个 notebook 适合在 VS Code / Jupyter 里做两件事：

1. 先检查目标文件夹里到底有多少张照片
2. 再调用本地加速版脚本批量提取 `User ID` 并生成 CSV


In [ ]:
from pathlib import Path
import csv
import subprocess
import sys

INPUT_DIR = Path('/Users/xiao1/Desktop/moomoo id')
SCRIPT_PATH = INPUT_DIR / 'extract_moomoo_id.py'
OUTPUT_CSV = INPUT_DIR / 'moomoo_user_ids.csv'

# 推荐参数：150 张照片先用 fast + 6 workers
MODE = 'fast'      # 可选: 'fast' / 'accurate'
WORKERS = 6        # 可根据机器性能调整，比如 4 / 6 / 8
RETRY_FAILED_WITH_ACCURATE = True

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.heic', '.heif'}


## 1. 扫描前检查文件夹

先运行下面这个 cell，确认照片数量和前几个文件名无误。

In [ ]:
image_files = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)

print(f'Folder: {INPUT_DIR}')
print(f'Total images: {len(image_files)}')
print('Head:')
for path in image_files[:20]:
    print('-', path.name)

print(f'\nMode: {MODE}')
print(f'Workers: {WORKERS}')
print(f'Retry failed with accurate: {RETRY_FAILED_WITH_ACCURATE}')

## 2. 调用加速版脚本生成 CSV

这个版本会直接调用 `extract_moomoo_id.py`，支持：

- `fast` 模式：更适合几百到上千张图片
- 多 worker 并行处理


In [ ]:
cmd = [
    'python3',
    str(SCRIPT_PATH),
    str(INPUT_DIR),
    '--csv-output',
    str(OUTPUT_CSV),
    '--mode',
    MODE,
    '--workers',
    str(WORKERS),
]

if RETRY_FAILED_WITH_ACCURATE:
    cmd.append('--retry-failed-with-accurate')

print('Running command:')
print(' '.join(cmd))

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()

if return_code not in (0, 2):
    raise RuntimeError(f'Command failed with exit code {return_code}')

## 3. 预览 CSV 前几行

In [ ]:
rows = []
with OUTPUT_CSV.open('r', encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        rows.append(row)

rows[:10]